In [1]:
import time
import re
import os
import platform
import pandas as pd
from datetime import datetime, timedelta

import pyautogui

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager    # 크롬 드라이버 자동 업데이트

from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver import ActionChains

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

from random import *
from pathlib import Path
from dateutil import parser

In [2]:
import pandas as pd
import requests


In [7]:
def read_discuss_list(fname):
    df = pd.read_csv(fname, delimiter='\t', encoding='utf-8', header=None, 
                    names=['ExamType', 'ExamNo', 'DiscussNo', 'DataID', 'PostDate', 'DiscussURL'], 
                    index_col=False)
    
    df['ExamType'] = df['ExamType'].str.strip()
    df['ExamNo'] = df['ExamNo'].astype(int)
    df['DiscussNo'] = df['DiscussNo'].astype(int)
    df['DataID'] = df['DataID'].astype(int)
    df['PostDate'] = pd.to_datetime(df['PostDate'], format='%Y-%m-%d %H:%M')
    df['DiscussURL'] = df['DiscussURL'].str.strip()

    df.drop_duplicates(inplace=True)
    return df.sort_values(by=['PostDate', 'DiscussNo'], ascending=[False, False])

def get_data_id(session, headers, discuss_id):
    url = f"https://www.examtopics.com/discussions/amazon/view/{discuss_id}-exam"

    resp = session.get(url, headers=headers)
    bs  = BeautifulSoup(resp.text, 'html.parser')
    elements = bs.select('[data-id]')
    for el in elements:
        if int(el['data-id']) > 0:
            return int(el['data-id'])
        
    return -1

def write_discuss_list(df, fname):
    df['PostDate'] = df.groupby(['ExamType', 'ExamNo', 'DiscussNo'])['PostDate'].transform('max')
    df.drop_duplicates(subset=['ExamType', 'ExamNo', 'DiscussNo'], keep='last', inplace=True)
    df['MaxDataID'] = df.groupby(['ExamType', 'ExamNo'])['DataID'].transform('max')
    df['Chk'] = df.apply(lambda row: 1 if row['DataID'] == row['MaxDataID'] else 0, axis=1)
    df = df.sort_values(['Chk', 'PostDate', 'DiscussNo'], ascending=[False, False, False]).drop(columns=['MaxDataID', 'Chk'])
    df.to_csv(fname, sep='\t', header=False, index=False)


In [8]:

discuss_list = 'AmzDiscuss.txt'
df = read_discuss_list(discuss_list)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Cookie": "csrftoken=xxxx; sessionid=xxxx;"   # 브라우저에서 복사한 쿠키 넣기
}

session = requests.Session()

for index in range(len(df)):
    if df.iloc[index]['DataID'] == 0:
        row = df.iloc[index]
        did = get_data_id(session, headers, row['DiscussNo'])
        row['DataID'] = did
        print(str(row['DiscussNo'])+"\t"+str(did), flush=True)

write_discuss_list(df, discuss_list)

152298	935345


C:\Users\Changwha\AppData\Local\Temp\ipykernel_1876\4148255764.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  row['DataID'] = did


312994	964393
315666	968355
315508	970212
315500	970203
150829	933873
150812	933856
302408	943733
152170	935285
312966	964401
311801	949526
305570	949525
313033	964377
312977	964354
131470	908904
304583	949521
305567	949520
305566	949519
150734	933839
304581	949511
312981	964388
150626	933829
151095	933850
133045	908898
313015	964404
312987	964402
313038	964400
150743	932379
150741	932380
312976	964349
312990	964424
312965	964422
312991	964419
312984	964415
150344	932363
150401	932353
150342	932350
312980	964341
312982	964365
312989	964361
312993	964357
313023	964344
312978	964360
312974	964358
142537	922731
142560	922730
132702	908973
78476	806968
312975	964378
313007	964386
168979	942063
152088	935275
168918	942060
168865	942057
168857	942047
312983	964373
312967	964369
312964	964368
306667	954888
306671	954877
306659	954880
313036	964362
47458	808823
312968	964359
306657	954866
80667	806952
78606	806951
78434	806950
308667	958783
152136	935268
146993	928540
304552	949508
150664	9338

In [4]:
len(df)

14243

In [3]:
for index in range(len(df)-1, -1, -1):
    row = df.iloc[index]
    # Your processing logic goes here
    print(f"Processing row {index}:", row)

Processing row 14242: ExamType      Exam AWS Certified Data Analytics - Specialty ...
ExamNo                                                       52
DiscussNo                                                 64622
DataID                                                        0
LastPost                                    2021-09-22 19:50:00
DiscussURL    /discussions/amazon/view/64622-exam-aws-certif...
Name: 14242, dtype: object
Processing row 14241: ExamType      Exam AWS Certified Solutions Architect - Assoc...
ExamNo                                                      434
DiscussNo                                                 61105
DataID                                                   719878
LastPost                                    2021-09-24 10:08:00
DiscussURL    /discussions/amazon/view/61105-exam-aws-certif...
Name: 14241, dtype: object
Processing row 14240: ExamType      Exam AWS Certified Solutions Architect - Profe...
ExamNo                                          